# exp_006_baseline_replay — Colab Dev-Split Runner

Runs inference on **`data/splits/dev.jsonl`** (200 questions) only — no private inference, no submission.csv.

**Before running:**
1. Runtime → Change runtime type → GPU (T4 / L4 / A100)
2. Run Cell 1 (install), then **Runtime → Restart runtime**
3. Run remaining cells in order

In [ ]:
## Cell 1 — Install packages  (restart runtime after this cell)
!pip install -q vllm transformers tqdm "antlr4-python3-runtime==4.11.1" "protobuf<6.0"
print("Install complete. → Runtime → Restart runtime, then run the next cell.")

In [ ]:
## Cell 2 — Clone repo
import os

REPO_DIR = "/content/151B_SP26_Competition"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Trevis8688/151B_SP26_Competition.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")

In [ ]:
## Cell 3 — Imports & configuration
import json
import importlib.util
import io as _io
import os
import re
import sys
import glob
from pathlib import Path
from typing import Optional

# Must be set BEFORE importing vllm
os.environ["VLLM_USE_V1"]                    = "0"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["PYTORCH_ALLOC_CONF"]             = "expandable_segments:True"

# Colab wraps stdout/stderr with virtual streams that raise UnsupportedOperation on fileno().
# vLLM calls fileno() internally — replace with real fd-backed streams first.
for _name, _fd in [('stdout', 1), ('stderr', 2)]:
    try:
        getattr(sys, _name).fileno()
    except _io.UnsupportedOperation:
        setattr(sys, _name, open(f'/dev/fd/{_fd}', 'w', buffering=1))

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

EXPERIMENT  = "exp_006_baseline_replay"
REPO_DIR    = "/content/151B_SP26_Competition"
WORKING_DIR = Path("/content")

# Load experiment config
_cfg_path = f"{REPO_DIR}/experiments/{EXPERIMENT}/config.json"
with open(_cfg_path) as _f:
    EXP_CONFIG = json.load(_f)
print(f"Loaded config from {_cfg_path}")

# Paths
MODEL_ID         = EXP_CONFIG.get("model_id", "Qwen/Qwen3-4B-Thinking-2507")
DATA_PATH        = f"{REPO_DIR}/data/splits/dev.jsonl"   # always dev on Colab
JUDGER_DIR       = REPO_DIR
PUBLIC_RESP_PATH = WORKING_DIR / "dev_responses.jsonl"
RESULTS_PATH     = WORKING_DIR / f"{EXPERIMENT}_results.jsonl"

# Sampling config
MAX_TOKENS  = EXP_CONFIG.get("max_tokens", 8192)
TEMPERATURE = EXP_CONFIG.get("temperature", 0.6)
TOP_P       = EXP_CONFIG.get("top_p", 0.95)
TOP_K       = EXP_CONFIG.get("top_k", 20)
NUM_SAMPLES = EXP_CONFIG.get("num_samples", 1)

# vLLM config
_vllm                   = EXP_CONFIG.get("vllm", {})
VLLM_MAX_MODEL_LEN      = _vllm.get("max_model_len", 10240)
VLLM_MAX_NUM_SEQS       = _vllm.get("max_num_seqs", 32)
VLLM_MAX_BATCHED_TOKENS = _vllm.get("max_num_batched_tokens", 20480)
VLLM_GPU_MEM_UTIL       = _vllm.get("gpu_memory_utilization", 0.90)

# Auto-detect dtype: A100/L4/H100 support bfloat16; T4/V100 need float16
import torch
_gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
_bf16_gpus = ["A100", "A10", "L4", "H100", "H200"]
DTYPE = "bfloat16" if any(x in _gpu_name for x in _bf16_gpus) else "float16"

N_GPUS = torch.cuda.device_count()
TENSOR_PARALLEL = 1  # change to 2 if you see 2 GPUs below

print(f"GPU        : {_gpu_name} ({N_GPUS} visible)")
print(f"dtype      : {DTYPE}")
print(f"Experiment : {EXPERIMENT}")
print(f"Model      : {MODEL_ID}")
print(f"Data       : {DATA_PATH}")
print(f"max_model_len={VLLM_MAX_MODEL_LEN}  max_num_seqs={VLLM_MAX_NUM_SEQS}  max_tokens={MAX_TOKENS}  split=dev")

In [ ]:
## Cell 4 — Load dev dataset
data = [json.loads(line) for line in open(DATA_PATH)]
n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options")   for d in data)
print(f"Loaded {len(data)} questions  ({n_mcq} MCQ, {n_free} free-form)")

In [ ]:
## Cell 5 — Load prompts
_prompts_path = f"{REPO_DIR}/experiments/{EXPERIMENT}/prompts.py"
_spec = importlib.util.spec_from_file_location("exp_prompts", _prompts_path)
_mod  = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
SYSTEM_PROMPT_MATH = _mod.SYSTEM_PROMPT_MATH
SYSTEM_PROMPT_MCQ  = _mod.SYSTEM_PROMPT_MCQ
FEWSHOT_MATH       = getattr(_mod, "FEWSHOT_MATH", [])
FEWSHOT_MCQ        = getattr(_mod, "FEWSHOT_MCQ",  [])
print(f"Loaded prompts from {_prompts_path}")
print(f"FEWSHOT_MCQ: {len(FEWSHOT_MCQ)} messages  FEWSHOT_MATH: {len(FEWSHOT_MATH)} messages")

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

In [ ]:
## Cell 6 — Load model with vLLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype=DTYPE,
    tensor_parallel_size=TENSOR_PARALLEL,
    enable_prefix_caching=False,
    gpu_memory_utilization=VLLM_GPU_MEM_UTIL,
    max_model_len=VLLM_MAX_MODEL_LEN,
    trust_remote_code=True,
    max_num_seqs=VLLM_MAX_NUM_SEQS,
    max_num_batched_tokens=VLLM_MAX_BATCHED_TOKENS,
)

sampling_params = SamplingParams(
    n=NUM_SAMPLES,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    min_p=0.0,
)
print("Model loaded.")

In [ ]:
## Cell 7 — Run inference on dev split
from collections import Counter

def extract_boxed(text):
    matches = re.findall(r'\\boxed\{([^}]*)\}', text or "")
    return matches[-1].strip() if matches else None

def majority_vote(texts):
    if len(texts) == 1:
        return texts[0]
    answers = [(extract_boxed(t), t) for t in texts]
    valid   = [(ans, t) for ans, t in answers if ans is not None]
    if not valid:
        return texts[0]
    modal = Counter(ans for ans, _ in valid).most_common(1)[0][0]
    return next(t for ans, t in valid if ans == modal)

def run_inference(items, out_path):
    prompts = []
    for item in items:
        system, user = build_prompt(item["question"], item.get("options"))
        fewshot = FEWSHOT_MCQ if item.get("options") else FEWSHOT_MATH
        messages = [{"role": "system", "content": system}, *fewshot, {"role": "user", "content": user}]
        prompts.append(tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        ))
    print(f"Generating {len(prompts)} responses (n={NUM_SAMPLES}) → {out_path} ...")
    outputs = llm.generate(prompts, sampling_params=sampling_params)

    responses = []
    for out in outputs:
        texts = [o.text.strip() for o in out.outputs]
        responses.append(majority_vote(texts))

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, "w") as f:
        for item, resp in zip(items, responses):
            f.write(json.dumps({
                "id": item["id"],
                "is_mcq": bool(item.get("options")),
                "response": resp,
            }) + "\n")
    print(f"Saved {len(responses)} responses to {out_path}")
    return responses

responses = run_inference(data, PUBLIC_RESP_PATH)

# Quick preview
for i in range(min(2, len(responses))):
    print(f"\n── Response {i} (id={data[i].get('id')}) ──")
    print(responses[i][:400], "..." if len(responses[i]) > 400 else "")

In [ ]:
## Cell 8 — Score results
if JUDGER_DIR not in sys.path:
    sys.path.insert(0, JUDGER_DIR)

judger = None
try:
    from judger import Judger
    judger = Judger(strict_extract=False)
    print("Judger loaded.")
except Exception as e:
    print(f"Could not load judger ({e}). MCQ-only scoring.")

def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""

results = []
for item, response in tqdm(zip(data, responses), total=len(data), desc="Scoring"):
    is_mcq = bool(item.get("options"))
    gold   = item["answer"]
    try:
        if is_mcq:
            correct = extract_letter(response) == str(gold).strip().upper()
        elif judger is not None:
            gold_list = gold if isinstance(gold, list) else [gold]
            correct = judger.auto_judge(pred=response, gold=gold_list, options=[[]] * len(gold_list))
        else:
            correct = None
    except Exception:
        correct = False
    results.append({"id": item.get("id"), "is_mcq": is_mcq, "gold": gold, "response": response, "correct": correct})

with open(RESULTS_PATH, "w") as f:
    for r in results:
        f.write(json.dumps(r) + "\n")
print(f"Saved {len(results)} records to {RESULTS_PATH}")

In [ ]:
## Cell 9 — Accuracy summary
mcq_res  = [r for r in results if     r["is_mcq"] and r["correct"] is not None]
free_res = [r for r in results if not r["is_mcq"] and r["correct"] is not None]
scored   = mcq_res + free_res

def acc(subset):
    return sum(bool(r["correct"]) for r in subset) / len(subset) * 100 if subset else 0.0

print("=" * 50)
print("LOCAL EVAL (dev split, 200 questions)")
print("=" * 50)
print(f"  MCQ        : {sum(bool(r['correct']) for r in mcq_res):4d} / {len(mcq_res):4d}  ({acc(mcq_res):.2f}%)")
print(f"  Free-form  : {sum(bool(r['correct']) for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
print(f"  Overall    : {sum(bool(r['correct']) for r in scored):4d} / {len(scored):4d}  ({acc(scored):.2f}%)")
print("=" * 50)
print(f"\nResults saved to: {RESULTS_PATH}")
print("Download via Files panel (left sidebar) to run /analyze locally.")